# Project #20 — Production-Grade AI Reasoning Agent\n\nThis notebook walks from foundations to production architecture using **LangGraph + Ollama + dynamic tools + memory + evaluation**.

## 1) What You Build\n\n- Planner / Executor / Tool Router / Observation Processor / Response Generator\n- ReAct loop with retries + reflection\n- Dynamic tool registry with schemas\n- Chroma semantic memory\n- Benchmark + LLM-as-a-Judge analytics

In [ ]:
from reasoning_agent.settings import load_settings\nfrom reasoning_agent.agent.runner import AgentRunner\nfrom reasoning_agent.schemas import ReasoningMode\nsettings = load_settings()\nsettings.model_dump()

## 2) Agent Lifecycle and ReAct Loop\n\nEach iteration tracks: **Thought -> Action -> Observation -> Reflection**.\nThe graph ends when objective is solved, max iterations reached, or recovery fails.

In [ ]:
runner = AgentRunner(settings=settings)\nresponse = runner.run(session_id='notebook-session', user_input='Calculate sin(0.5) + log(10) and explain the steps.', mode=ReasoningMode.REACT)\nresponse.answer

In [ ]:
response.metrics, response.termination_reason, len(response.trace)

## 3) Tool Registry and Dynamic Discovery\n\nNo hardcoded if/else chains. Tools self-register with metadata + input/output schemas.

In [ ]:
tools = runner.artifacts.registry.metadata()\nlen(tools), tools[:3]

## 4) Semantic Memory with Chroma\n\nConversation and observations are persisted. Retrieval can ground future responses.

In [ ]:
hits = runner.artifacts.memory.retrieve('sin log calculation', k=3)\n[(h.score, h.text[:120]) for h in hits]

## 5) Benchmarking 100+ Prompts\n\nRuns across Qwen/Llama/Granite/DeepSeek-R1 when available. Missing models are auto-skipped with explicit reason.

In [ ]:
from reasoning_agent.evaluation import BenchmarkRunner, BenchmarkVisualizer\nbench = BenchmarkRunner(settings)\nresult = bench.run()\nsummary = result['summary']\nsummary

In [ ]:
viz = BenchmarkVisualizer('artifacts/plots')\nviz.generate_all(summary)

## 6) Observability and Trace Files\n\nEach run emits structured logs and JSON traces for postmortem and analytics.

In [ ]:
import pathlib\ntrace_files = sorted(pathlib.Path(settings.log_dir).joinpath('traces').glob('*.json'))\ntrace_files[-3:] if trace_files else []

## 7) Streamlit Product Interface\n\nLaunch app with:\n```bash\nuv run streamlit run streamlit_app/Home.py\n```\nPages: Chat, Execution Trace, Tool Calls, Memory, Analytics, Settings.

## 8) Limitations and Next Improvements\n\n- Quality depends on local model performance.\n- Network tools require internet/provider reliability.\n- Optional phase-2 features: async/parallel tools, streaming tokens, MCP adapter, human approval checkpoints.